# Multi-head attention
1. Two heads by hand on "money bank" (4 numbers per word): two sets of W_Q, W_K, W_V, concatenate, W_O.
2. The paper's version (512 numbers, 8 heads of 64) by hand in NumPy, checked against keras.layers.MultiHeadAttention.
3. Parameter and work counts: 8 heads of 64 cost the same as 1 head of 512.
4. A real trained model: the 12 heads of BERT-base's first layer on "the man saw the astronomer with a telescope".

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import json, struct, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import keras

np.set_printoptions(precision=3, suppress=True)


def softmax_rows(S):
    W = np.exp(S - S.max(-1, keepdims=True))
    return W / W.sum(-1, keepdims=True)


def attention(Q, K, V):
    """Scaled dot-product attention: one head."""
    A = softmax_rows(Q @ K.T / np.sqrt(K.shape[1]))
    return A @ V, A

## 1. Two heads by hand: "money bank", 4 numbers per word
Each head has its own 4x4 W_Q, W_K, W_V (as in the simplest picture of multi-head attention).

In [2]:
rng = np.random.default_rng(0)
X = np.array([[1.0, 0.0, 1.0, 0.0],      # money
              [0.0, 2.0, 0.0, 1.0]])     # bank
heads = [{m: rng.integers(-1, 2, (4, 4)).astype(float) for m in ("Q", "K", "V")} for _ in range(2)]
Z, A = [], []
for i, W in enumerate(heads, 1):
    Q, K, V = X @ W["Q"], X @ W["K"], X @ W["V"]
    z, a = attention(Q, K, V)
    Z.append(z); A.append(a)
    print(f"head {i}: W_Q, W_K, W_V =\n{W['Q']}\n{W['K']}\n{W['V']}")
    print(f"head {i}: Q =\n{Q}\nK =\n{K}\nscores/2 =\n{Q @ K.T / 2}\nweights =\n{a}\nZ{i} =\n{z}\n")
Zcat = np.concatenate(Z, axis=1)
WO = rng.integers(-1, 2, (8, 4)).astype(float)
Zout = Zcat @ WO
print("concatenated Z' shape", Zcat.shape, "\n", Zcat)
print("W_O shape", WO.shape, "\n", WO)
print("final output shape", Zout.shape, "\n", Zout)

head 1: W_Q, W_K, W_V =
[[ 1.  0.  0. -1.]
 [-1. -1. -1. -1.]
 [-1.  1.  0.  1.]
 [ 0.  0.  1.  1.]]
[[ 0.  0.  0.  1.]
 [-1.  1.  1. -1.]
 [ 0.  1.  0. -1.]
 [ 1.  1.  1. -1.]]
[[-1.  1. -1.  0.]
 [-1. -1.  0.  0.]
 [ 0. -1. -1. -1.]
 [-1.  1.  0.  0.]]
head 1: Q =
[[ 0.  1.  0.  0.]
 [-2. -2. -1. -1.]]
K =
[[ 0.  1.  0.  0.]
 [-1.  3.  3. -3.]]
scores/2 =
[[ 0.5  1.5]
 [-1.  -2. ]]
weights =
[[0.269 0.731]
 [0.731 0.269]]
Z1 =
[[-2.462 -0.731 -0.538 -0.269]
 [-1.538 -0.269 -1.462 -0.731]]

head 2: W_Q, W_K, W_V =
[[-1.  0.  1.  0.]
 [ 0.  1.  1.  1.]
 [ 0.  1.  1.  0.]
 [ 1.  1.  1.  0.]]
[[ 1. -1.  0.  1.]
 [ 1.  0.  0. -1.]
 [ 0.  0.  1.  1.]
 [-1.  1.  0.  0.]]
[[ 1.  0. -1. -1.]
 [ 1.  0.  0.  0.]
 [ 1.  0. -1.  1.]
 [-1. -1.  1.  0.]]
head 2: Q =
[[-1.  1.  2.  0.]
 [ 1.  3.  3.  2.]]
K =
[[ 1. -1.  1.  2.]
 [ 1.  1.  0. -2.]]
scores/2 =
[[0.  0. ]
 [2.5 0. ]]
weights =
[[0.5   0.5  ]
 [0.924 0.076]]
Z2 =
[[ 1.5   -0.5   -0.5    0.   ]
 [ 1.924 -0.076 -1.772  0.   ]]

concatenat

## 2. The paper's version: d_model = 512, h = 8 heads of d_k = d_v = 64
By hand in NumPy, then the same with Keras' MultiHeadAttention using the same weights.

In [3]:
words = "the man saw the astronomer with a telescope".split()
n, d, h = len(words), 512, 8
dk = d // h
keras.utils.set_random_seed(0)
X = np.random.default_rng(1).normal(size=(1, n, d)).astype("float32")   # stand-in embeddings
mha = keras.layers.MultiHeadAttention(num_heads=h, key_dim=dk)
out_keras, w_keras = mha(X, X, return_attention_scores=True)
Wq, bq, Wk, bk, Wv, bv, Wo, bo = mha.get_weights()
print("Keras weight shapes:", [w.shape for w in mha.get_weights()])

x = X[0]
Zs, As = [], []
for i in range(h):                                   # one head at a time
    Q = x @ Wq[:, i] + bq[i]                         # (n, 64)
    K = x @ Wk[:, i] + bk[i]
    V = x @ Wv[:, i] + bv[i]
    z, a = attention(Q, K, V)
    Zs.append(z); As.append(a)
Zcat = np.concatenate(Zs, axis=1)                    # (n, 512)
out_hand = Zcat @ Wo.reshape(h * dk, d) + bo         # W_O: (512, 512)
print("shapes: each head", Zs[0].shape, " concatenated", Zcat.shape, " output", out_hand.shape)
print("largest difference, outputs:", np.abs(out_hand - out_keras.numpy()[0]).max())
print("largest difference, weights:", np.abs(np.array(As) - w_keras.numpy()[0]).max())
assert np.allclose(out_hand, out_keras.numpy()[0], atol=1e-4)
assert np.allclose(np.array(As), w_keras.numpy()[0], atol=1e-5)
print("different heads, different weights: row of 'man' in heads 1 and 2")
print(As[0][1], "\n", As[1][1])

Keras weight shapes: [(512, 8, 64), (8, 64), (512, 8, 64), (8, 64), (512, 8, 64), (8, 64), (8, 64, 512), (512,)]
shapes: each head (8, 64)  concatenated (8, 512)  output (8, 512)
largest difference, outputs: 6.470746423126172e-07
largest difference, weights: 3.6094865274449717e-07
different heads, different weights: row of 'man' in heads 1 and 2
[0.206 0.089 0.099 0.146 0.04  0.18  0.137 0.104] 
 [0.092 0.097 0.419 0.128 0.023 0.104 0.058 0.078]


## 3. Same cost: 8 heads of 64 vs 1 head of 512

In [4]:
rows = []
for heads_, key_dim in ((1, 512), (8, 64), (2, 512)):
    m = keras.layers.MultiHeadAttention(num_heads=heads_, key_dim=key_dim)
    m(X, X)
    rows.append(dict(heads=heads_, size_per_head=key_dim, parameters=m.count_params(),
                     score_mult_adds_per_pair=heads_ * key_dim))
cost = pd.DataFrame(rows)
print(cost.to_string(index=False))
print("4 d^2 + 4 d =", 4 * d * d + 4 * d)
cost.to_csv("data/param_count.csv", index=False)

 heads  size_per_head  parameters  score_mult_adds_per_pair
     1            512     1050624                       512
     8             64     1050624                       512
     2            512     2100736                      1024
4 d^2 + 4 d = 1050624


## 4. A real trained model: BERT-base, layer 1 (the first), 12 heads
Only the weights needed for the attention weights of the first layer are downloaded (byte ranges of the
official safetensors file, about 7 MB) and cached. BERT-base: 12 layers, 768 numbers per token, 12 heads of 64.

In [5]:
URL = "https://huggingface.co/bert-base-uncased/resolve/main/"
cache = Path.home() / ".keras" / "datasets" / "bert_base_uncased_layer0.npz"
toks = ["[CLS]"] + words + ["[SEP]"]
if not cache.exists():
    def get_bytes(a, b):
        req = urllib.request.Request(URL + "model.safetensors", headers={"Range": f"bytes={a}-{b}"})
        return urllib.request.urlopen(req).read()
    hlen = struct.unpack("<Q", get_bytes(0, 7))[0]
    header = json.loads(get_bytes(8, 8 + hlen - 1))
    def tensor(name, rows=None):
        s, e = header[name]["data_offsets"]
        shape = header[name]["shape"]
        s += 8 + hlen
        if rows is None:
            return np.frombuffer(get_bytes(s, s + e - header[name]["data_offsets"][0] - 1), "<f4").reshape(shape)
        w = shape[1] * 4
        return np.stack([np.frombuffer(get_bytes(s + r * w, s + (r + 1) * w - 1), "<f4") for r in rows])
    vocab = urllib.request.urlopen(URL + "vocab.txt").read().decode().split("\n")
    ids = [vocab.index(t) for t in toks]
    e, q = "bert.embeddings.", "bert.encoder.layer.0.attention.self."
    np.savez(cache, ids=ids,
             word=tensor(e + "word_embeddings.weight", ids), pos=tensor(e + "position_embeddings.weight", range(len(ids))),
             typ=tensor(e + "token_type_embeddings.weight", [0]), ln_g=tensor(e + "LayerNorm.gamma"),
             ln_b=tensor(e + "LayerNorm.beta"), Wq=tensor(q + "query.weight"), bq=tensor(q + "query.bias"),
             Wk=tensor(q + "key.weight"), bk=tensor(q + "key.bias"))
B = np.load(cache)
print("token ids:", dict(zip(toks, B["ids"])))
# embedding = word + position + segment, then layer normalisation (BERT's input)
x = B["word"] + B["pos"] + B["typ"]
x = (x - x.mean(1, keepdims=True)) / np.sqrt(x.var(1, keepdims=True) + 1e-12) * B["ln_g"] + B["ln_b"]
Q = x @ B["Wq"].T + B["bq"]                           # PyTorch stores weights as (out, in)
K = x @ B["Wk"].T + B["bk"]
A_bert = np.array([softmax_rows(Q[:, i*64:(i+1)*64] @ K[:, i*64:(i+1)*64].T / 8) for i in range(12)])
print("shape (heads, query word, key word):", A_bert.shape)

token ids: {'[CLS]': np.int64(101), 'the': np.int64(1996), 'man': np.int64(2158), 'saw': np.int64(2387), 'astronomer': np.int64(15211), 'with': np.int64(2007), 'a': np.int64(1037), 'telescope': np.int64(12772), '[SEP]': np.int64(102)}
shape (heads, query word, key word): (12, 10, 10)


The same weights through Keras' layer (the value and output matrices do not affect the attention weights).

In [6]:
bert_mha = keras.layers.MultiHeadAttention(num_heads=12, key_dim=64)
bert_mha(x[None].astype("float32"), x[None].astype("float32"))
w = bert_mha.get_weights()
w[0], w[1] = B["Wq"].T.reshape(768, 12, 64), B["bq"].reshape(12, 64)
w[2], w[3] = B["Wk"].T.reshape(768, 12, 64), B["bk"].reshape(12, 64)
bert_mha.set_weights(w)
_, A_keras = bert_mha(x[None].astype("float32"), x[None].astype("float32"), return_attention_scores=True)
print("largest difference NumPy vs Keras:", np.abs(A_bert - A_keras.numpy()[0]).max())
assert np.allclose(A_bert, A_keras.numpy()[0], atol=1e-5)

largest difference NumPy vs Keras: 4.7683716e-07


Where do "man" and "astronomer" look, head by head?

In [7]:
lab = [f"{t}_{i}" for i, t in enumerate(toks)]
for word_i, word in ((2, "man"), (5, "astronomer")):
    tab = pd.DataFrame(A_bert[:, word_i, :].round(2), columns=lab, index=[f"head {i+1}" for i in range(12)])
    print(f"\nrow of '{word}' in each head:\n", tab.to_string())


row of 'man' in each head:
          [CLS]_0  the_1  man_2  saw_3  the_4  astronomer_5  with_6   a_7  telescope_8  [SEP]_9
head 1      0.11   0.03   0.07   0.13   0.03          0.15    0.05  0.03         0.29     0.11
head 2      0.06   0.02   0.05   0.07   0.02          0.44    0.01  0.01         0.20     0.12
head 3      0.17   0.72   0.01   0.01   0.04          0.01    0.03  0.01         0.00     0.01
head 4      0.07   0.81   0.07   0.02   0.01          0.01    0.00  0.01         0.00     0.00
head 5      0.23   0.08   0.04   0.20   0.08          0.09    0.04  0.07         0.11     0.06
head 6      0.30   0.06   0.04   0.10   0.08          0.19    0.03  0.04         0.06     0.11
head 7      0.03   0.01   0.22   0.12   0.01          0.37    0.03  0.02         0.14     0.03
head 8      0.22   0.01   0.05   0.09   0.02          0.12    0.06  0.02         0.26     0.15
head 9      0.02   0.03   0.02   0.12   0.06          0.34    0.04  0.03         0.25     0.08
head 10     0.11   0.

A simple profile of each head (averaged over the 8 real words): weight on the next word, the previous word,
itself, and the two special tokens.

In [8]:
idx = np.arange(1, n + 1)
prof = pd.DataFrame({
    "next word": [A_bert[i, idx[:-1], idx[:-1] + 1].mean() for i in range(12)],
    "previous word": [A_bert[i, idx, idx - 1].mean() for i in range(12)],
    "itself": [A_bert[i, idx, idx].mean() for i in range(12)],
    "[CLS] or [SEP]": [(A_bert[i, idx, 0] + A_bert[i, idx, -1]).mean() for i in range(12)],
}, index=[f"head {i+1}" for i in range(12)]).round(2)
print(prof.to_string())
pd.DataFrame([dict(head=i + 1, query=toks[r], query_pos=r, key=toks[c], key_pos=c, weight=float(A_bert[i, r, c]))
              for i in range(12) for r in range(len(toks)) for c in range(len(toks))]).to_csv(
    "data/bert_layer1_attention.csv", index=False, float_format="%.5f")

         next word  previous word  itself  [CLS] or [SEP]
head 1        0.09           0.07    0.12            0.17
head 2        0.07           0.10    0.08            0.16
head 3        0.03           0.38    0.06            0.48
head 4        0.03           0.49    0.08            0.09
head 5        0.08           0.08    0.08            0.27
head 6        0.06           0.10    0.06            0.32
head 7        0.08           0.08    0.17            0.17
head 8        0.04           0.08    0.10            0.43
head 9        0.11           0.07    0.07            0.18
head 10       0.05           0.08    0.15            0.35
head 11       0.60           0.05    0.02            0.20
head 12       0.07           0.21    0.04            0.42
